# Student Stress: Domain Ablation, SHAP ও Counterfactual Analysis
Baseline: Gradient Boosting | Data: `StressLevelDataset.csv` (1100 × 21)

**RQ:** (1) কোন domain সবচেয়ে গুরুত্বপূর্ণ? (2) domain বাদ দিলে performance কতটা কমে? (3) কোন feature বদলালে model-এর predicted stress কমে? (4) profile অনুযায়ী পার্থক্য আছে কি?

**নোট:** Data observational, তাই সব ফল "model অনুযায়ী", causal নয়।

In [ ]:
# Step 1: imports and config
import os, json, itertools
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RepeatedStratifiedKFold, cross_validate, GridSearchCV)
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             matthews_corrcoef, roc_auc_score, confusion_matrix)
from sklearn.preprocessing import label_binarize

try:
    import shap
except ImportError:
    shap = None  # pip install shap

sns.set_style("whitegrid")
os.makedirs("outputs", exist_ok=True)

DATA_PATH, TARGET, SEED = "StressLevelDataset.csv", "stress_level", 42
N_SPLITS, N_REPEATS = 5, 3
MAX_STEP, MAX_CHANGED = 2, 3
SHAP_BACKGROUND, SHAP_PERM = 50, 5
RUN_NESTED_CV = True

## ১. Data ও Domain mapping

In [ ]:
# Step 2: load, audit, domain mapping
df = pd.read_csv(DATA_PATH)
print(df.shape, "| missing:", int(df.isna().sum().sum()), "| duplicates:", int(df.duplicated().sum()))
print(df[TARGET].value_counts(normalize=True).sort_index().round(3).to_dict())

DOMAINS = {
    "psychological": ["anxiety_level", "self_esteem", "mental_health_history", "depression"],
    "physiological": ["headache", "blood_pressure", "sleep_quality", "breathing_problem"],
    "academic": ["academic_performance", "study_load", "future_career_concerns"],
    "social": ["teacher_student_relationship", "social_support", "peer_pressure",
               "extracurricular_activities", "bullying"],
    "environmental": ["noise_level", "living_conditions", "safety", "basic_needs"],
}
# feature -> improving direction (+1: higher is better, -1: lower is better)
MODIFIABLE = {"sleep_quality": 1, "social_support": 1, "study_load": -1,
              "extracurricular_activities": 1, "noise_level": -1, "living_conditions": 1}

FEATURES = [c for c in df.columns if c != TARGET]
mapped = [f for fs in DOMAINS.values() for f in fs]
assert sorted(mapped) == sorted(FEATURES), "domain mapping does not match dataset columns"
assert set(MODIFIABLE) <= set(FEATURES)
F2D = {f: d for d, fs in DOMAINS.items() for f in fs}

## ২. Split ও direction check
Holdout শুধু final report ও explanation-এ ব্যবহার হবে। Direction উল্টো (`⚠️`) এলে `MODIFIABLE`-এর ওই feature ঠিক করুন।

In [ ]:
# Step 3: train/holdout split and direction check
X, y = df[FEATURES], df[TARGET]
X_train, X_hold, y_train, y_hold = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
CLASSES = sorted(y.unique())
HIGH = max(CLASSES)

check = pd.DataFrame({
    "feature": list(MODIFIABLE),
    "rho": [stats.spearmanr(X_train[f] * d, y_train)[0] for f, d in MODIFIABLE.items()],
})
check["ok"] = np.where(check["rho"] < 0, "OK", "⚠️")
display(check.round(3))

## ৩. Baseline (Gradient Boosting) ও Nested CV

In [ ]:
# Step 4: helpers
def make_pipeline(**params):
    return Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("clf", GradientBoostingClassifier(random_state=SEED, **params))])

def full_metrics(y_true, y_pred, y_prob):
    yb = label_binarize(y_true, classes=CLASSES)
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Acc": balanced_accuracy_score(y_true, y_pred),
        "F1-weighted": f1_score(y_true, y_pred, average="weighted"),
        "F1-macro": f1_score(y_true, y_pred, average="macro"),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "ROC-AUC": roc_auc_score(yb, y_prob, multi_class="ovr", average="weighted"),
        "Brier": float(np.mean(np.sum((yb - y_prob) ** 2, axis=1))),
    }

def holm(p):
    p = np.asarray(p, float); order = np.argsort(p); adj = np.empty(len(p)); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (len(p) - rank) * p[i]); adj[i] = min(run, 1.0)
    return adj

In [ ]:
# Step 5: baseline fit and holdout evaluation
model = make_pipeline().fit(X_train, y_train)
hold_pred, hold_prob = model.predict(X_hold), model.predict_proba(X_hold)
baseline = full_metrics(y_hold, hold_pred, hold_prob)
display(pd.DataFrame([baseline], index=["GB holdout"]).round(4))

sns.heatmap(confusion_matrix(y_hold, hold_pred, labels=CLASSES), annot=True, fmt="d",
            cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Holdout confusion matrix"); plt.show()
json.dump({k: round(float(v), 5) for k, v in baseline.items()},
          open("outputs/baseline_results.json", "w"), indent=2)

In [ ]:
# Step 6: nested CV (outer 5, inner 3)
nested_f1 = None
if RUN_NESTED_CV:
    grid = {"clf__n_estimators": [100, 200], "clf__max_depth": [2, 3],
            "clf__learning_rate": [0.05, 0.1]}
    search = GridSearchCV(make_pipeline(), grid, scoring="f1_weighted", n_jobs=-1,
                          cv=StratifiedKFold(3, shuffle=True, random_state=SEED))
    nested_f1 = cross_validate(search, X_train, y_train, scoring="f1_weighted",
                               cv=StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED))["test_score"]
    print(f"Nested CV F1-weighted: {nested_f1.mean():.4f} ± {nested_f1.std():.4f}")

## ৪. Domain Ablation
সব condition-এ একই CV split ও default hyperparameter। `- domain` = সেই domain বাদ, `only domain` = শুধু সেটি। Repeated-CV fold স্বাধীন নয়, তাই p-value কিছুটা optimistic।

In [ ]:
# Step 7: domain ablation
rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
conditions = {"All features": FEATURES}
conditions.update({f"- {d}": [f for f in FEATURES if f not in fs] for d, fs in DOMAINS.items()})
conditions.update({f"only {d}": fs for d, fs in DOMAINS.items()})

fold_f1, rows = {}, []
for name, cols in conditions.items():
    cv = cross_validate(make_pipeline(), X_train[cols], y_train, cv=rskf,
                        scoring="f1_weighted", n_jobs=-1)
    fold_f1[name] = cv["test_score"]
    hold_f1 = f1_score(y_hold, make_pipeline().fit(X_train[cols], y_train).predict(X_hold[cols]),
                       average="weighted")
    rows.append({"Condition": name, "n_features": len(cols), "CV F1": cv["test_score"].mean(),
                 "CV SD": cv["test_score"].std(), "Holdout F1": hold_f1})

ablation = pd.DataFrame(rows)
ref = ablation.loc[0, "CV F1"]
ablation["ΔF1 vs All"] = ablation["CV F1"] - ref
display(ablation.round(4))
ablation.round(5).to_csv("outputs/ablation_results.csv", index=False)

In [ ]:
# Step 8: Friedman and Wilcoxon (Holm)
names = ["All features"] + [f"- {d}" for d in DOMAINS]
fr = stats.friedmanchisquare(*[fold_f1[n] for n in names])
print(f"Friedman: chi2={fr.statistic:.3f}, p={fr.pvalue:.4g}")

raw = []
for n in names[1:]:
    try:
        raw.append(stats.wilcoxon(fold_f1["All features"], fold_f1[n]).pvalue)
    except ValueError:
        raw.append(1.0)
tests = pd.DataFrame({"Condition": names[1:],
                      "mean ΔF1": [(fold_f1["All features"] - fold_f1[n]).mean() for n in names[1:]],
                      "raw p": raw})
tests["Holm p"] = holm(tests["raw p"])
tests["significant"] = tests["Holm p"] < 0.05
display(tests.round(4))
tests.round(5).to_csv("outputs/ablation_stats.csv", index=False)

In [ ]:
# Step 8: ablation plot
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
m = ablation[ablation["Condition"].str.startswith("- ")].assign(drop=lambda d: -d["ΔF1 vs All"]).sort_values("drop")
ax[0].barh(m["Condition"], m["drop"], color="tab:red")
ax[0].set_title("F1 drop when domain removed")
o = ablation[ablation["Condition"].str.startswith("only ")].sort_values("CV F1")
ax[1].barh(o["Condition"], o["CV F1"], xerr=o["CV SD"], color="tab:blue")
ax[1].axvline(ref, color="k", ls="--")
ax[1].set_title("CV F1 using one domain only")
plt.tight_layout(); plt.savefig("outputs/ablation_plot.png", dpi=150); plt.show()

## ৫. SHAP
Multiclass `GradientBoostingClassifier`-এ `TreeExplainer` চলে না, তাই Permutation explainer (probability space, approximate)।

In [ ]:
# Step 9: SHAP on holdout
shap_vals = None
if shap is not None:
    def predict_fn(a):
        return model.predict_proba(pd.DataFrame(a, columns=FEATURES))

    bg = shap.sample(X_train, SHAP_BACKGROUND, random_state=SEED)
    explainer = shap.Explainer(predict_fn, bg, algorithm="permutation",
                               max_evals=2 * len(FEATURES) * SHAP_PERM + 1)
    shap_vals = np.asarray(explainer(X_hold.values, silent=True).values)  # (n, features, classes)

    class_cols = [f"class {c}" for c in model.named_steps["clf"].classes_]
    imp = pd.DataFrame(np.abs(shap_vals).mean(axis=0), index=FEATURES, columns=class_cols)
    imp["overall"] = imp.mean(axis=1)
    imp = imp.sort_values("overall", ascending=False)
    display(imp.head(10).round(4))

    dom_share = imp["overall"].groupby(imp.index.map(F2D)).sum()
    dom_share = (dom_share / dom_share.sum() * 100).sort_values(ascending=False)
    display(dom_share.round(2).to_frame("SHAP share %"))

    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    imp.head(15).iloc[::-1][class_cols].plot(kind="barh", stacked=True, ax=ax[0])
    ax[0].set_title("Top 15 features (mean |SHAP|)")
    dom_share.iloc[::-1].plot(kind="barh", color="tab:green", ax=ax[1])
    ax[1].set_title("Domain share (%)")
    plt.tight_layout(); plt.savefig("outputs/shap.png", dpi=150); plt.show()
    imp.round(5).to_csv("outputs/shap_importance.csv")

## ৬. Constrained Counterfactual
শুধু `MODIFIABLE` feature, শুধু ভালো দিকে, প্রতি feature সর্বোচ্চ `MAX_STEP` ধাপ, মোট সর্বোচ্চ `MAX_CHANGED`টা feature। সফল হলে সবচেয়ে কম feature ও কম ধাপের সমাধান নেওয়া হয়। বাকি feature স্থির ধরা হয় (বাস্তবে correlated), তাই এটি model-based hypothetical।

In [ ]:
# Step 10: single-feature what-if (students predicted High)
classes_ = list(model.named_steps["clf"].classes_)
IDX_HIGH = classes_.index(HIGH)
RANGES = {f: (int(X_train[f].min()), int(X_train[f].max())) for f in MODIFIABLE}
pred_s = pd.Series(hold_pred, index=X_hold.index)
high = X_hold[pred_s == HIGH]
print("Predicted High in holdout:", len(high))

rows = []
for f, d in MODIFIABLE.items():
    lo, hi = RANGES[f]
    for s in range(1, MAX_STEP + 1):
        new = high[f] + d * s
        ok = new.between(lo, hi)
        if not ok.any():
            continue
        base = high[ok]; mod = base.assign(**{f: new[ok]})
        rows.append({"feature": f, "step": s, "n": int(ok.sum()),
                     "mean ΔP(High)": (model.predict_proba(mod)[:, IDX_HIGH]
                                       - model.predict_proba(base)[:, IDX_HIGH]).mean(),
                     "% class lowered": (model.predict(mod) < HIGH).mean() * 100})
whatif = pd.DataFrame(rows).sort_values(["step", "mean ΔP(High)"])
display(whatif.round(4))
whatif.round(5).to_csv("outputs/whatif.csv", index=False)

whatif.pivot(index="feature", columns="step", values="mean ΔP(High)").plot(kind="barh", figsize=(7, 3.5))
plt.axvline(0, color="k", lw=0.8); plt.title("Mean ΔP(High) per single-feature change")
plt.tight_layout(); plt.savefig("outputs/whatif.png", dpi=150); plt.show()

In [ ]:
# Step 11: minimal counterfactual search
col_idx = {c: i for i, c in enumerate(FEATURES)}

def candidates(row):
    opts = {}
    for f, d in MODIFIABLE.items():
        lo, hi = RANGES[f]
        vals = [row[f] + d * s for s in range(1, MAX_STEP + 1) if lo <= row[f] + d * s <= hi]
        if vals:
            opts[f] = vals
    out = []
    for k in range(1, MAX_CHANGED + 1):
        for subset in itertools.combinations(opts, k):
            for vals in itertools.product(*[opts[f] for f in subset]):
                out.append(dict(zip(subset, vals)))
    return out

def counterfactual(row, orig):
    cands = candidates(row)
    if not cands:
        return None
    arr = np.tile(row.values.astype(float), (len(cands), 1))
    for i, ch in enumerate(cands):
        for f, v in ch.items():
            arr[i, col_idx[f]] = v
    proba = model.predict_proba(pd.DataFrame(arr, columns=FEATURES))
    new_cls = np.array(classes_)[proba.argmax(axis=1)]
    io = classes_.index(orig)
    ok = np.where(new_cls < orig)[0]
    if len(ok) == 0:
        return None
    best = min(ok, key=lambda i: (len(cands[i]),
                                  sum(abs(v - row[f]) for f, v in cands[i].items()),
                                  proba[i, io]))
    p0 = model.predict_proba(row.to_frame().T.astype(float))[0, io]
    return {"changes": cands[best], "n_changed": len(cands[best]),
            "new_class": int(new_cls[best]), "p_before": float(p0), "p_after": float(proba[best, io])}

rows = []
for pos, (idx, row) in enumerate(X_hold.iterrows()):
    orig = int(hold_pred[pos])
    if orig == min(CLASSES):
        continue
    res = counterfactual(row, orig)
    rows.append({"pos": pos, "index": idx, "orig_class": orig, "found": res is not None, **(res or {})})
cf = pd.DataFrame(rows)

display(cf.groupby("orig_class").agg(students=("found", "size"), found=("found", "sum"))
          .assign(success_pct=lambda d: (d["found"] / d["students"] * 100).round(1)))
found = cf[cf["found"]]
if len(found):
    display(found["n_changed"].value_counts().sort_index().to_frame("students"))
    freq = pd.Series(Counter(f for ch in found["changes"] for f in ch)).sort_values()
    freq.plot(kind="barh", figsize=(6, 3), color="tab:purple")
    plt.title("Features changed in minimal counterfactuals")
    plt.tight_layout(); plt.savefig("outputs/cf_frequency.png", dpi=150); plt.show()
cf.drop(columns=["changes"], errors="ignore").to_csv("outputs/counterfactual.csv", index=False)

## ৭. Profile analysis
প্রতিটা predicted-High student-এর dominant profile = যে domain-এর SHAP contribution (High class-এ) সবচেয়ে বেশি। n < 5 হলে ফল দুর্বল।

In [ ]:
# Step 12: profile-wise counterfactual comparison
profile_table = None
if shap_vals is not None:
    d_idx = {d: [FEATURES.index(f) for f in fs] for d, fs in DOMAINS.items()}
    prof = pd.DataFrame([
        {"pos": int(p), "profile": max(d_idx, key=lambda d: shap_vals[p, d_idx[d], IDX_HIGH].sum())}
        for p in np.where(hold_pred == HIGH)[0]])
    g_all = cf[cf["orig_class"] == HIGH].merge(prof, on="pos")

    out = []
    for p, g in g_all.groupby("profile"):
        gf = g[g["found"]]
        top = Counter(f for ch in gf["changes"] for f in ch).most_common(3)
        out.append({"profile": p, "n": len(g),
                    "CF success %": round(len(gf) / len(g) * 100, 1),
                    "mean features changed": round(gf["n_changed"].mean(), 2) if len(gf) else np.nan,
                    "mean P(High) drop": round((gf["p_before"] - gf["p_after"]).mean(), 3) if len(gf) else np.nan,
                    "top features": ", ".join(f"{f}({c})" for f, c in top)})
    profile_table = pd.DataFrame(out).sort_values("n", ascending=False)
    display(profile_table)
    profile_table.to_csv("outputs/profiles.csv", index=False)

## ৮. Summary

In [ ]:
# Step 13: summary
worst = ablation[ablation["Condition"].str.startswith("- ")].sort_values("ΔF1 vs All").iloc[0]
print(f"Baseline holdout: Acc={baseline['Accuracy']:.4f}, F1w={baseline['F1-weighted']:.4f}, MCC={baseline['MCC']:.4f}")
if nested_f1 is not None:
    print(f"Nested CV F1w: {nested_f1.mean():.4f} ± {nested_f1.std():.4f}")
print(f"Largest ablation drop: {worst['Condition']} (ΔF1={worst['ΔF1 vs All']:+.4f}) | Friedman p={fr.pvalue:.4g}")
print("Significant after Holm:", tests.loc[tests["significant"], "Condition"].tolist() or "none")
if shap_vals is not None:
    print(f"Top SHAP domain: {dom_share.index[0]} ({dom_share.iloc[0]:.1f}%)")
print(f"Counterfactual success (predicted Medium/High): {cf['found'].mean() * 100:.1f}%")